# 📡 EcoRoute AI — Objective 2: IoT Sensor Monitoring & Threshold-Based Classification
### Ultrasonic Telemetry Processing, Anomaly Filtering & Dynamic Status Engine
---
**Project:** EcoRoute AI (Smart IoT Waste Management System)  
**Objective 2:** Ingest continuous IoT sensor streams from ESP32 DevKit V1 and HC-SR04 ultrasonic nodes, apply noise filtration, calculate accurate volumetric fill percentage, and classify dustbins into real-time operational hazard states:  
- 🟢 **Normal:** Fill Level $< 60\%$ (No immediate collection required)  
- 🟡 **Nearly Full:** Fill Level $60\% - 79\%$ (Scheduled for upcoming collection cycle)  
- 🔴 **Critical Overflow Alert:** Fill Level $\ge 80\%$ (Audible alarm triggered + Emergency dispatch priority)  

**Hardware Target:** Google Colab (Python 3.10+ / Pandas / NumPy / Scikit-Learn)

## Step 1: Install Dependencies & Mount Google Drive

In [ ]:
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn

from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EcoRoute_AI/Objective2_IoT'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('📁 Google Drive Workspace ready at:', DRIVE_DIR)

## Step 2: Realistic IoT Sensor Stream Simulation (ESP32 + HC-SR04)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
N_SAMPLES = 2500

BIN_IDS = [f'DB-{i:03d}' for i in range(1, 16)]
timestamps = pd.date_range(start='2026-11-01 06:00:00', periods=N_SAMPLES, freq='5min')

bin_height = 40.0

raw_distances = np.random.uniform(2.0, 42.0, N_SAMPLES)
noise = np.random.normal(0, 1.2, N_SAMPLES)
measured_distances = np.clip(raw_distances + noise, 2.0, 45.0)

spike_indices = np.random.choice(N_SAMPLES, size=int(N_SAMPLES * 0.04), replace=False)
measured_distances[spike_indices] = np.random.choice([0.5, 48.0, 99.0], size=len(spike_indices))

df_iot = pd.DataFrame({
    'timestamp': timestamps,
    'bin_id': np.random.choice(BIN_IDS, N_SAMPLES),
    'bin_depth_cm': bin_height,
    'raw_distance_cm': np.round(measured_distances, 2),
    'battery_voltage_v': np.round(np.random.uniform(3.5, 4.2, N_SAMPLES), 2),
    'rssi_dbm': np.random.randint(-85, -45, N_SAMPLES)
})

csv_raw_path = os.path.join(DRIVE_DIR, 'raw_iot_telemetry.csv')
df_iot.to_csv(csv_raw_path, index=False)
print(f'✅ Generated {N_SAMPLES} realistic IoT telemetry readings. Saved to {csv_raw_path}')
df_iot.head()

## Step 3: Anomaly Filtering & Volumetric Fill-Level Calculation

In [ ]:
df_clean = df_iot.copy()
df_clean = df_clean[(df_clean['raw_distance_cm'] >= 2.0) & (df_clean['raw_distance_cm'] <= 42.0)].copy()

df_clean['filtered_distance_cm'] = df_clean.groupby('bin_id')['raw_distance_cm'].transform(
    lambda s: s.rolling(window=3, min_periods=1).median()
)

df_clean['fill_level_pct'] = np.clip(
    ((df_clean['bin_depth_cm'] - df_clean['filtered_distance_cm']) / df_clean['bin_depth_cm']) * 100.0,
    0.0, 100.0
)

def classify_status(fill):
    if fill >= 80.0:
        return 'Critical Overflow'
    elif fill >= 60.0:
        return 'Nearly Full'
    else:
        return 'Normal'

df_clean['status'] = df_clean['fill_level_pct'].apply(classify_status)

csv_clean_path = os.path.join(DRIVE_DIR, 'processed_iot_classification.csv')
df_clean.to_csv(csv_clean_path, index=False)

print('✅ Anomaly filtering and status classification complete!')
print(df_clean['status'].value_counts())

## Step 4: Machine Learning Classifier Benchmark (Random Forest vs Threshold)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

X = df_clean[['filtered_distance_cm', 'battery_voltage_v', 'rssi_dbm']]
y = df_clean['status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'✅ ML Validation Accuracy: {acc*100:.2f}%')
print('\n' + '='*60)
print('📊 SENSOR CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_test, y_pred, digits=4))

rf_model_path = os.path.join(DRIVE_DIR, 'sensor_status_classifier.joblib')
joblib.dump(rf_classifier, rf_model_path)
print(f'Model saved at: {rf_model_path}')

## Step 5: IoT Telemetry Visualizations & Analytics Dashboard

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 10))

plt.subplot(2, 2, 1)
palette = {'Normal': '#10b981', 'Nearly Full': '#f59e0b', 'Critical Overflow': '#ef4444'}
sns.histplot(data=df_clean, x='fill_level_pct', hue='status', palette=palette, kde=True, bins=35)
plt.title('Fill Level Distribution & Safety Thresholds', fontsize=12, fontweight='bold')
plt.xlabel('Volumetric Fill Level (%)')
plt.ylabel('Observation Count')

plt.subplot(2, 2, 2)
sample_bins = df_clean['bin_id'].unique()[:4]
sample_ts = df_clean[df_clean['bin_id'].isin(sample_bins)].sort_values('timestamp').tail(150)
sns.lineplot(data=sample_ts, x='timestamp', y='fill_level_pct', hue='bin_id', marker='o')
plt.axhline(y=80, color='#ef4444', linestyle='--', label='80% Overflow Alarm')
plt.axhline(y=60, color='#f59e0b', linestyle=':', label='60% Warning Threshold')
plt.title('Real-Time IoT Telemetry Stream (Last 150 Cycles)', fontsize=12, fontweight='bold')
plt.xlabel('Timestamp')
plt.ylabel('Fill Level (%)')
plt.xticks(rotation=25)
plt.legend(loc='lower left')

plt.subplot(2, 2, 3)
subset_noise = df_clean.head(80)
plt.plot(subset_noise['raw_distance_cm'].values, label='Raw Sensor Echo (With Jitter)', color='#ef4444', alpha=0.6)
plt.plot(subset_noise['filtered_distance_cm'].values, label='Filtered Distance (Rolling Median)', color='#10b981', lw=2)
plt.title('Acoustic Echo Jitter Filtration', fontsize=12, fontweight='bold')
plt.xlabel('Sample Index')
plt.ylabel('Distance to Trash (cm)')
plt.legend()

plt.subplot(2, 2, 4)
counts = df_clean['status'].value_counts()
colors = [palette[k] for k in counts.index]
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=[0.05]*len(counts))
plt.title('Fleet Dustbin State Breakdown', fontsize=12, fontweight='bold')

plt.tight_layout()
dashboard_path = os.path.join(DRIVE_DIR, 'iot_telemetry_analytics.png')
plt.savefig(dashboard_path, dpi=300)
plt.show()

print(f'✅ IoT Analytics Dashboard saved at: {dashboard_path}')
print('\n🎉 OBJECTIVE 2 COMPLETED!')